# Category Classification — Inference Pipeline

**Responsibility:** download the registered model from the Hopsworks Model Registry → apply feature engineering → produce predictions. No training happens here. Runs without any upstream variables in memory.

| Input | Description |
|---|---|
| Hopsworks Model Registry `category_classifier` | Best model by macro F1 |
| New CSV (same schema as training data) | Rows to score |

| Output | Description |
|---|---|
| `category_predictions.csv` | Predicted category, confidence, and per-class probabilities |

> **Prerequisites:** Training Pipeline (`category_classification_t.ipynb`) must have run and uploaded a model to the Hopsworks Model Registry.

## Shared Imports & Configuration

Load core libraries (numpy, pandas, matplotlib) and the Hopsworks client.
All display and warning settings are tuned for a clean notebook output.
The Inference Pipeline has no training dependencies — it only needs the
model artifact and the TOML configuration to reproduce feature engineering.


In [ ]:
import os
import pickle
import random
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import hopsworks
import tomllib
from category_classification import (
    CategoryPredictor,
    find_latest_sor_date, iter_sor_chunks,
    compute_macro_prf, draw_stream_dashboard,
)
import io
import datetime
import time
import gc
from collections import deque
import ipywidgets as widgets
import requests
import remotezip
from IPython.display import clear_output, display


# Silence non-critical warnings so the stream output stays readable.
warnings.filterwarnings("ignore")
# Widen the terminal display so wide DataFrames don't get truncated.
pd.set_option("display.max_columns", 50)
print("Imports OK.")


## TOML Configuration

Load `category_classification_fti.toml` and import inference-specific helpers
from `category_classification.py` (CategoryPredictor, stream probes, dashboard).
The config drives every downstream constant: paths, Hopsworks credentials,
stream parameters, and feature-engineering defaults.


In [ ]:
# CategoryPredictor wraps the loaded artifact + local feature engineering.
# Stream helpers probe CloudFront zips and render the live dashboard.
# (all imported in the first cell above)

# Single source of truth: edit this TOML to change paths, credentials,
# hyper-parameters, or feature lists across all three pipeline stages.
CFG_PATH = Path("category_classification_fti.toml")
with open(CFG_PATH, "rb") as f:
    cfg = tomllib.load(f)

print(f"Config loaded from {CFG_PATH}")


## Path, Constant & Service Definitions

Resolve all file-system paths from config, pin the random seed, and wire up
Hopsworks connection parameters. Every downstream cell reads from these
constants — edit the TOML to change them.

> **Note:** The Inference Pipeline does not need the Training Pipeline's
> variables in memory. It is fully self-contained from this cell onward.


In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
_paths = cfg["paths"]
MODEL_ARTIFACT    = Path(_paths["model_artifact"])     # pickle loaded in IP-1
MODEL_EXPORT_DIR  = Path(_paths["model_export_dir"])   # staging before Hopsworks upload
CV_RESULTS_PATH   = Path(_paths["cv_results"])         # hyper-param search log (unused here)
PREDICTIONS_PATH  = Path(_paths["predictions"])        # batch inference output (unused here)
METADATA_PATH     = Path(_paths["metadata"])           # label maps & feature lists

# ── Reproducibility ────────────────────────────────────────────────────────────
_repr = cfg["reproducibility"]
RANDOM_STATE  = _repr["random_state"]   # seeds numpy, random, sklearn, LightGBM
N_CV_FOLDS    = _repr["n_cv_folds"]     # not used in inference, kept for consistency
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

# ── Domain constants ───────────────────────────────────────────────────────────
_domain = cfg["domain"]
OTHER_CAT  = _domain["other_category"]   # catch-all class used by TwoStageClassifier
PREFIX     = _domain["prefix"]           # stripped from DSA category strings
TEXT_COL   = _domain["text_col"]         # primary policy text (Tier-2/3)
TEXT_COL_2 = _domain["text_col_2"]       # free-text rationale (Tier-2/3)
TEXT_COL_3 = _domain["text_col_3"]       # decision facts (Tier-3 only)
HIGH_CARD  = cfg["features"]["categorical"]["encoding"]["high_cardinality"]

# ── Hopsworks ─────────────────────────────────────────────────────────────────
# Environment variables override TOML so secrets never need to be committed.
_hw = cfg["hopsworks"]
HOPSWORKS_HOST    = os.environ.get("HOPSWORKS_HOST",    _hw["host"])
HOPSWORKS_PROJECT = os.environ.get("HOPSWORKS_PROJECT", _hw["project"])
HOPSWORKS_API_KEY = os.environ.get("HOPSWORKS_API_KEY", _hw["api_key"])
FG_NAME           = _hw["feature_group"]["name"]
FG_VERSION        = _hw["feature_group"]["version"]
FG_TEXT_1_NAME    = _hw.get("feature_group_text_1", {}).get("name",    FG_NAME + "_text_1")
FG_TEXT_1_VERSION = _hw.get("feature_group_text_1", {}).get("version", 1)
FG_TEXT_2_NAME    = _hw.get("feature_group_text_2", {}).get("name",    FG_NAME + "_text_2")
FG_TEXT_2_VERSION = _hw.get("feature_group_text_2", {}).get("version", 1)
FG_TEXT_3_NAME    = _hw.get("feature_group_text_3", {}).get("name",    FG_NAME + "_text_3")
FG_TEXT_3_VERSION = _hw.get("feature_group_text_3", {}).get("version", 1)
FV_NAME           = _hw.get("feature_view", {}).get("name",    FG_NAME + "_view")
FV_VERSION        = _hw.get("feature_view", {}).get("version", 1)
MODEL_NAME        = _hw["model_registry"]["name"]

print(f"Metadata  : {METADATA_PATH}")
print(f"Model     : {MODEL_ARTIFACT}")
print(f"Hopsworks : {HOPSWORKS_HOST}  project={HOPSWORKS_PROJECT}")
print(f"  FG (struct) : {FG_NAME} v{FG_VERSION}")
print(f"  FG (text 1) : {FG_TEXT_1_NAME} v{FG_TEXT_1_VERSION}")
print(f"  FG (text 2) : {FG_TEXT_2_NAME} v{FG_TEXT_2_VERSION}")
print(f"  FG (text 3) : {FG_TEXT_3_NAME} v{FG_TEXT_3_VERSION}")
print(f"  FV          : {FV_NAME} v{FV_VERSION}")
print(f"  Model       : {MODEL_NAME}")


---
# Part 3 — Inference Pipeline

**Responsibility:** download the registered model from the Hopsworks Model Registry →
apply local feature engineering → produce predictions. No training happens here.
Runs without any upstream variables in memory.

**Input:** Hopsworks Model Registry entry `category_classifier` (best by macro F1) +
a new CSV (same schema as training data).  
**Output:** `category_predictions.csv` (batch) or live stream CSV (streaming).


## IP-1 · Download Model from Hopsworks Model Registry

Connect to Hopsworks, then either reuse the local `category_best_model.pkl`
(fast path) or download the best model from the registry. The artifact is
unpickled into a dict and wrapped by `CategoryPredictor`, which re-implements
the training-era feature engineering for fully local scoring.


In [ ]:
# Connect to Hopsworks using the same credentials as the Feature / Training pipelines.
project = hopsworks.login(
    host=HOPSWORKS_HOST,
    project=HOPSWORKS_PROJECT,
    api_key_value=HOPSWORKS_API_KEY,
)
mr = project.get_model_registry()
print(f"Connected  : {HOPSWORKS_HOST}  →  project '{project.name}'")

# ── Download or use cached model ───────────────────────────────────────────────
# The Training Pipeline writes category_best_model.pkl locally.  If it exists we
# skip the Hopsworks download to save time and bandwidth.
if MODEL_ARTIFACT.exists():
    print(f"\nUsing cached local model: {MODEL_ARTIFACT}")
    _model_path = MODEL_ARTIFACT
else:
    # Fetch the best model by macro F1 from the registry and cache it locally.
    best_model_meta = mr.get_best_model(MODEL_NAME, metric="macro_f1", direction="max")
    print(f"\nDownloading '{MODEL_NAME}' v{best_model_meta.version}  "
          f"(macro_f1={best_model_meta.training_metrics.get('macro_f1', 'n/a')})")
    model_dir   = Path(best_model_meta.download())
    _model_path = model_dir / MODEL_ARTIFACT.name
    shutil.copy(_model_path, MODEL_ARTIFACT)
    print(f"Cached locally → {MODEL_ARTIFACT}")

# Unpickle the artifact dict (keys: pipeline, preprocessors, label maps, …).
with open(_model_path, "rb") as f:
    ma = pickle.load(f)

# CategoryPredictor re-implements the training-era feature engineering so we
# can score new rows without any Feature Store connection.
predictor = CategoryPredictor(ma, cfg)
infer_target_labels   = predictor.target_labels
infer_model_name      = predictor.model_name
infer_tier            = predictor.tier

print(f"\nModel: {infer_model_name}  (Tier {infer_tier})")
print(f"Classes ({len(infer_target_labels)}): {infer_target_labels[:5]} …")


## IP-2 · Prediction Function

The actual scoring method is `predictor.predict(df_raw, history_df=None)`.
It applies feature engineering and runs the loaded model in one call.
See `CategoryPredictor` in `category_classification.py` for the full implementation.


In [ ]:
# predictor.predict(df_raw) — local scoring (no Feature Store)
# Usage: pass a raw DataFrame with the same schema as training data.
# Optional history_df warm-starts rolling-window features for accuracy.


## IP-3 · Configure Stream Parameters

Define stream-control widgets (delay, max rows, display columns), read defaults
from the `[inference]` TOML section, and probe the CloudFront index to find
the most recent available date.

> **Tip:** Reduce *Delay* to 0.05 s for a fast simulation, or raise it to 1.0 s
> to read each row comfortably.


In [ ]:


# Pull defaults from the TOML [fa0] and [inference] sections.
_fa0_cfg     = cfg["fa0"]
_infer_cfg   = cfg.get("inference", {})
_IS4_BASE    = _fa0_cfg["cloudfront_base"]
_IS4_SUBPATH = _fa0_cfg["cloudfront_subpath"]

# Full schema of columns we may want to display in the live feed.
_IS4_ALL_COLS = [
    "platform_name", "created_at", "decision_ground", "category",
    "decision_visibility", "content_type", "decision_account",
    "decision_provision", "decision_monetary", "automated_detection",
    "automated_decision", "source_type", "content_language",
    "territorial_scope", "application_date",
]
# Default display subset — kept narrow so the terminal feed is readable.
_IS4_DEFAULT_COLS = _infer_cfg.get("stream_default_columns", [
    "platform_name", "created_at", "decision_ground",
    "content_type", "decision_visibility",
])

# Delay between rows — lower = faster animation, higher = easier to read.
_is4_delay = widgets.FloatSlider(
    value=_infer_cfg.get("stream_delay_seconds", 0.3), min=0.0, max=5.0, step=0.05,
    description="Delay (s):", readout_format=".2f",
    layout=widgets.Layout(width="420px"),
)
# Hard cap on rows to score — protects against multi-million-row chunks.
_is4_max_rows = widgets.BoundedIntText(
    value=_infer_cfg.get("stream_max_rows", 3000), min=1, max=10_000, step=10,
    description="Max rows:",
    layout=widgets.Layout(width="200px"),
)
# Multi-select for which columns appear in the live feed.
_is4_cols_widget = widgets.SelectMultiple(
    options=_IS4_ALL_COLS, value=_IS4_DEFAULT_COLS,
    description="Columns:",
    layout=widgets.Layout(width="360px", height="200px"),
)
display(widgets.VBox([
    widgets.HBox([_is4_delay, _is4_max_rows]),
    _is4_cols_widget,
    widgets.Label("Adjust parameters, then run Fetch and Stream cells below."),
]))


## IP-4 · Fetch Latest Stream Data

Probe the CloudFront index for the most recent parquet zip, fetch the last
`stream_tail_chunks` chunks via HTTP range requests (no full-zip download),
and write each chunk to a temporary parquet file on disk.  The concatenated
span is checked using only the first and last chunk timestamps so the full
dataset never resides in RAM.  Resolve the display column list.


In [ ]:
# Probe CloudFront for the most recent daily parquet zip (looks back 10 days).
import tempfile, gc
from pathlib import Path

print("Probing for most recent available parquet zip...")
_stream_date = find_latest_sor_date(_IS4_BASE, _IS4_SUBPATH)
print(f"  Most recent date: {_stream_date}\n")

print("Fetching chunks via HTTP range requests...")
_WARMUP_HOURS = cfg["feature_pipeline"].get("rolling_window_hours", 3)
_tail = _infer_cfg.get("stream_tail_chunks", 2)

# Create a temp directory for disk-cached chunks so memory stays bounded.
_chunk_dir = Path(tempfile.mkdtemp(prefix="stream_chunks_"))
_chunk_files = []  # list of (name, path)
_total_rows = 0
_stream_chunk_name = ""

while True:
    # Reset timestamp bounds on every iteration so earlier chunks are discovered.
    _first_ts = None
    _last_ts  = None
    _current_chunk_files = []
    _current_names = []

    for name, df in iter_sor_chunks(_stream_date, _IS4_BASE, _IS4_SUBPATH, tail=_tail):
        p = _chunk_dir / f"{name}"
        df.to_parquet(p, index=False)
        _current_chunk_files.append((name, p))
        _current_names.append(name)
        _total_rows += len(df)

        # Track timestamp bounds across ALL chunks in this pass.
        _ts_col_probe = "created_at" if "created_at" in df.columns else "application_date"
        _ts_probe = pd.to_datetime(df[_ts_col_probe], errors="coerce").dropna()
        if len(_ts_probe):
            if _first_ts is None or _ts_probe.min() < _first_ts:
                _first_ts = _ts_probe.min()
            if _last_ts is None or _ts_probe.max() > _last_ts:
                _last_ts = _ts_probe.max()
        del df
        gc.collect()

    _chunk_files = _current_chunk_files
    _stream_chunk_name = ", ".join(_current_names)

    # Compute span from the true min / max timestamps seen this pass.
    if _first_ts is not None and _last_ts is not None:
        _span_hours = (_last_ts - _first_ts).total_seconds() / 3600
    else:
        _span_hours = 0

    if _span_hours >= _WARMUP_HOURS or _tail >= 20:
        break

    _tail += 2
    print(f"  Span {_span_hours:.1f}h < {_WARMUP_HOURS}h — increasing to {_tail} chunks...")

# Peek at the first chunk's schema to resolve display columns.
import pyarrow.parquet as pq
_probe_cols = list(pq.read_schema(_chunk_files[0][1]).names) if _chunk_files else []
_is4_cols = (
    [c for c in _is4_cols_widget.value if c in _probe_cols]
    if "_is4_cols_widget" in vars() and _probe_cols
    else [c for c in ["platform_name", "created_at", "decision_ground",
                       "content_type", "decision_visibility"]
          if c in _probe_cols]
)

print(f"\nCached {_total_rows:,} rows from {_stream_chunk_name} to disk.")
print(f"Displaying columns: {_is4_cols}")


## IP-5 · Define Inference Loop

Define `run_stream_inference_disk()`: streams each chunk in small batches
(5 000 rows) so a full ~1 M-row chunk is never materialised in RAM.
Rolling history is kept with only the four columns required for platform
features, and `engineer_features` mutates batches in-place to avoid copies.

> **Performance note:** Peak memory is bounded by ~5 000 rows + a 4-hour
> history window (~tens of MB), regardless of chunk count or chunk size.


In [ ]:
# Resolve UI / config parameters used by the inference loop.
_stream_max_rows = _is4_max_rows.value if "_is4_max_rows" in vars() else _infer_cfg.get("stream_max_rows", 3000)
_stream_delay    = _is4_delay.value if "_is4_delay" in vars() else _infer_cfg.get("stream_delay_seconds", 0.3)
_GRAPH_EVERY     = _infer_cfg.get("stream_graph_every", 20)
_FEED_LINES      = _infer_cfg.get("stream_feed_lines", 14)
_COL_MAX_CHARS   = _infer_cfg.get("stream_column_max_chars", 18)
_selected_cols   = _is4_cols if "_is4_cols" in vars() else _IS4_DEFAULT_COLS

# Class bookkeeping for one-vs-rest TP/FP/FN counters.
_classes   = infer_target_labels
_n_classes = len(_classes)
_cls_idx   = {lbl: i for i, lbl in enumerate(_classes)}

_out = widgets.Output()
display(_out)


def _trim_history(df, ts_col, hours):
    """Keep only rows within the last `hours` of the max timestamp."""
    if df is None or len(df) == 0:
        return df
    ts = pd.to_datetime(df[ts_col], errors="coerce")
    if ts.notna().any():
        cutoff = ts.max() - pd.Timedelta(hours=hours)
        mask = ts >= cutoff
        if mask.all():
            return df
        return df.loc[mask].copy()
    return df


def run_stream_inference_disk(
    chunk_files,
    _cls_idx, _n_classes,
    _selected_cols, _GRAPH_EVERY, _FEED_LINES, _out,
    _stream_date, _stream_chunk_name, _stream_delay,
    predictor, PREFIX,
    _WARMUP_HOURS, _stream_max_rows,
):
    """Score rows chunk-by-chunk from disk, keeping bounded history in memory.

    Processes each chunk in small batches so a full ~1 M-row chunk is never
    materialised in RAM.  History is kept with only the four columns required
    for rolling features.
    """
    import pyarrow.parquet as pq
    from tqdm.auto import tqdm
    from category_classification import safe_div

    # Self-contained accumulators — fresh on every call so re-running IP-6
    # does not aggregate metrics from previous runs.
    _tp = np.zeros(_n_classes, dtype=np.int64)
    _fp = np.zeros(_n_classes, dtype=np.int64)
    _fn = np.zeros(_n_classes, dtype=np.int64)

    _history_df = None
    _first_ts = None
    _n_scored = 0
    _n_labeled = 0
    _prediction_errors = 0
    _row_log = deque(maxlen=_FEED_LINES * 2)
    _hist_n  = []
    _hist_f1 = []
    _hist_pr = []
    _hist_re = []

    import csv
    _ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    _csv_path = Path(f"stream_inference_{_ts}.csv")

    _HIDE_COLS = {"decision_ground", "decision_visibility"}
    _display_cols = [c for c in _selected_cols if c not in _HIDE_COLS]

    _csv_fh = None
    _csv_writer = None
    _BATCH_SIZE = 20000

    print(f"[stream] Starting inference on {len(chunk_files)} chunk(s), max_rows={_stream_max_rows}")
    print(f"[stream] {_n_classes} classes: {list(_cls_idx.keys())[:5]}...")

    _warmup_batches_seen = 0
    _total_warmup_batches = None  # discovered dynamically

    for chunk_idx, (chunk_name, chunk_path) in enumerate(chunk_files, 1):
        print(f"[stream] Chunk {chunk_idx}/{len(chunk_files)}: {chunk_name}")
        pf = pq.ParquetFile(chunk_path)
        n_batches = sum(1 for _ in pf.iter_batches(batch_size=_BATCH_SIZE))
        pf = pq.ParquetFile(chunk_path)

        _batch_iter = list(pf.iter_batches(batch_size=_BATCH_SIZE))
        _n_total_batches = len(_batch_iter)

        # Determine warm-up batches with a quick scan (no scoring, just timestamps).
        _warmup_batch_indices = set()
        _tmp_first_ts = None
        for _tmp_idx, _tmp_batch in enumerate(_batch_iter, 1):
            _tmp_df = _tmp_batch.to_pandas()
            _tmp_ts_col = "created_at" if "created_at" in _tmp_df.columns else "application_date"
            _tmp_ts = pd.to_datetime(_tmp_df[_tmp_ts_col], errors="coerce")
            if _tmp_ts.notna().any():
                _tmp_min = _tmp_ts.min()
                if _tmp_first_ts is None or _tmp_min < _tmp_first_ts:
                    _tmp_first_ts = _tmp_min
            if _tmp_first_ts is not None:
                _tmp_cutoff = _tmp_first_ts + pd.Timedelta(hours=_WARMUP_HOURS)
                if (_tmp_ts < _tmp_cutoff).any():
                    _warmup_batch_indices.add(_tmp_idx)
            del _tmp_df
        gc.collect()

        _warmup_pbar = None
        for batch_idx, batch in enumerate(_batch_iter, 1):
            is_warmup = batch_idx in _warmup_batch_indices

            if is_warmup:
                if _warmup_pbar is None:
                    _warmup_pbar = tqdm(
                        total=len(_warmup_batch_indices),
                        desc="warm-up batches",
                        unit="batch",
                        leave=False,
                    )
                _warmup_pbar.update(1)
            elif _warmup_pbar is not None:
                _warmup_pbar.close()
                _warmup_pbar = None

            chunk_batch = batch.to_pandas()
            ts_col = "created_at" if "created_at" in chunk_batch.columns else "application_date"
            chunk_ts = pd.to_datetime(chunk_batch[ts_col], errors="coerce")

            if chunk_ts.notna().any():
                batch_min = chunk_ts.min()
                if _first_ts is None or batch_min < _first_ts:
                    _first_ts = batch_min

            if _first_ts is not None:
                _warmup_cutoff = _first_ts + pd.Timedelta(hours=_WARMUP_HOURS)
                score_mask = chunk_ts >= _warmup_cutoff
            else:
                score_mask = pd.Series(False, index=chunk_batch.index)

            n_scoreable = int(score_mask.sum())

            

            if _csv_fh is None and n_scoreable > 0:
                _csv_cols = ["predicted", "actual", "confidence", "correct"] + list(chunk_batch.columns)
                _csv_fh = open(_csv_path, "w", newline="", encoding="utf-8")
                _csv_writer = csv.writer(_csv_fh)
                _csv_writer.writerow(_csv_cols)
                print(f"\n[stream] Exporting to {_csv_path}")
                print("\033[1m[stream] See output above for inference (it can take a few moments for output to appear)\033[0m")

            # ── Update rolling history (minimal columns only) ──
            _hist_cols = [ts_col, "platform_name", "automated_detection", "content_type"]
            _avail_hist = [c for c in _hist_cols if c in chunk_batch.columns]
            if _avail_hist:
                _hist_batch = chunk_batch[_avail_hist].copy()
                if _history_df is None:
                    _history_df = _hist_batch
                else:
                    _history_df = pd.concat([_history_df, _hist_batch], ignore_index=True)
                    _history_df = _trim_history(_history_df, ts_col, _WARMUP_HOURS + 1)
                del _hist_batch
                gc.collect()

            # ── Score rows in this batch ──
            if n_scoreable > 0:
                score_batch = chunk_batch[score_mask]
                X_eng, texts_icg, texts_ice, texts_df = predictor.engineer_features(
                    score_batch, history_df=_history_df, inplace=True
                )

                for i in range(len(score_batch)):
                    if _n_scored >= _stream_max_rows:
                        break

                    try:
                        pred = predictor._score_features(
                            X_eng.iloc[[i]],
                            texts_icg.iloc[[i]],
                            texts_ice.iloc[[i]],
                            texts_df.iloc[[i]],
                            pd.Index([i]),
                        )
                        pred_cat   = pred["predicted_category"].iloc[0]
                        confidence = pred["confidence"].iloc[0]
                    except Exception as exc:
                        pred_cat   = "ERROR"
                        confidence = 0.0
                        _prediction_errors += 1
                        print(f"\n[stream] Row {_n_scored} predict error: {exc}")

                    actual_raw = score_batch.iloc[i].get("category", None)
                    actual_cat = (
                        str(actual_raw).replace(PREFIX, "")
                        if pd.notna(actual_raw) and actual_raw
                        else None
                    )
                    correct = (pred_cat == actual_cat) if actual_cat is not None else None

                    if actual_cat in _cls_idx and pred_cat in _cls_idx:
                        pi = _cls_idx[pred_cat]
                        ai = _cls_idx[actual_cat]
                        if pi == ai:
                            _tp[pi] += 1
                        else:
                            _fp[pi] += 1
                            _fn[ai] += 1
                        _n_labeled += 1

                    _n_scored += 1
                    n_done = _n_scored

                    _row_log.append({
                        "predicted":  pred_cat,
                        "actual":     actual_cat,
                        "confidence": confidence,
                        "correct":    correct,
                        "col_vals":   [str(score_batch.iloc[i].get(c, "—"))[:_COL_MAX_CHARS] for c in _display_cols],
                    })

                    if _csv_writer is not None:
                        _csv_writer.writerow(
                            [pred_cat, actual_cat or "", confidence, correct]
                            + [score_batch.iloc[i].get(c, "") for c in score_batch.columns]
                        )
                        _csv_fh.flush()

                    # Redraw dashboard every _GRAPH_EVERY rows (or at the end).
                    if n_done % _GRAPH_EVERY == 0 or n_done == _stream_max_rows:
                        mp, mr, mf = compute_macro_prf(_tp, _fp, _fn)
                        _hist_n.append(n_done)
                        _hist_f1.append(mf)
                        _hist_pr.append(mp)
                        _hist_re.append(mr)

                        # Build per-class diagnostic string for the first few classes.
                        _diag_lines = []
                        for lbl, idx in _cls_idx.items():
                            t, f, n = int(_tp[idx]), int(_fp[idx]), int(_fn[idx])
                            if t + f + n > 0:
                                p = safe_div(t, t + f)
                                r = safe_div(t, t + n)
                                f1c = safe_div(2 * p * r, p + r)
                                _diag_lines.append(f"    {lbl:<40s} TP={t:<4d} FP={f:<4d} FN={n:<4d} F1={f1c:.3f}")
                        _diag_str = "\n".join(_diag_lines[:8])

                        fig = draw_stream_dashboard(
                            n_done, _tp, _fp, _fn,
                            _hist_n, _hist_f1, _hist_pr, _hist_re,
                            _stream_date, _stream_chunk_name,
                        )
                        is_final = _n_scored >= _stream_max_rows

                        with _out:
                            clear_output(wait=True)
                            if is_final:
                                total_tp = int(_tp.sum()); total_fp = int(_fp.sum())
                                total_fn = int(_fn.sum())
                                print(f"Stream complete — {_n_scored:,} rows  ·  {_stream_date}  ·  {_stream_chunk_name}")
                                print(f"Macro F1={mf:.4f}  Precision={mp:.4f}  Recall={mr:.4f}")
                                print(f"TP={total_tp:,}  FP={total_fp:,}  FN={total_fn:,}")
                                if _prediction_errors:
                                    print(f"Prediction errors: {_prediction_errors:,}")
                                print("\nPer-class metrics:")
                                print(_diag_str)
                            else:
                                print(f"DSA Live Inference  ·  {_stream_date}  ·  {_stream_chunk_name}")
                                print(f"Row {n_done:,} / {_stream_max_rows:,}"
                                      f"   |   Macro F1 {mf:.3f}   P {mp:.3f}   R {mr:.3f}")
                                print("\nPer-class metrics (top 8):")
                                print(_diag_str)
                            col_hdrs = "  ".join(f"{c:<18}" for c in _display_cols)
                            print(f"\n  {'':<4}  {'predicted':<32}  {'actual':<32}  {'conf':<5}  {col_hdrs}")
                            print("─" * (78 + 20 * len(_display_cols)))
                            for entry in reversed(list(_row_log)[-_FEED_LINES:]):
                                sym = "✅" if entry["correct"] is True else ("❌" if entry["correct"] is False else "  ")
                                actual_str = entry["actual"] or "—"
                                cols_str = "  ".join(entry["col_vals"])
                                print(f"  {sym}  {entry['predicted']:<32}  {actual_str:<32}  {entry['confidence']:.3f}  {cols_str}")
                            print()
                            display(fig)
                        fig.clf()
                        plt.close(fig)

                    if _stream_delay > 0:
                        time.sleep(_stream_delay)

                del X_eng, texts_icg, texts_ice, texts_df, score_batch
                gc.collect()

            del chunk_batch
            gc.collect()

            if _n_scored >= _stream_max_rows:
                
                break

        chunk_path.unlink()

        if _n_scored >= _stream_max_rows:
            break

    

    if _csv_fh is not None:
        _csv_fh.close()
        print(f"[stream] Saved {_n_labeled:,} labeled rows to {_csv_path}")
        if _prediction_errors:
            print(f"[stream] Prediction errors: {_prediction_errors:,} row(s) excluded from metrics")
    else:
        print(f"[stream] No rows scored (warmup={_WARMUP_HOURS}h, first_ts={_first_ts}, chunks={len(chunk_files)})")


## IP-6 · Run Stream Inference

Call `run_stream_inference_disk()` with the parameters set above to simulate real-time
streaming from the DSA CloudFront archive. The loop scores each row live, updates
the dashboard every `stream_graph_every` rows, appends all results to a timestamped
CSV, and prints a summary when the stream completes.

> **Note: streaming only begins after processing all warm-up batches.**

> **Why simulated streaming?** The live DSA research API requires registration
> with a statement of intent. Using the published daily parquet zips lets anyone
> reproduce the demo without API credentials.


In [ ]:
# Kick off the live inference loop with all parameters resolved above.
# Accumulators (_tp, _fp, _fn) are created fresh inside the function so
# re-running this cell does not aggregate metrics from previous runs.
run_stream_inference_disk(
    _chunk_files,
    _cls_idx, _n_classes,
    _selected_cols, _GRAPH_EVERY, _FEED_LINES, _out,
    _stream_date, _stream_chunk_name, _stream_delay,
    predictor, PREFIX,
    _WARMUP_HOURS, _stream_max_rows,
)
